# VISCERA — FAIR test of concept-supervised pretraining (30k bundle)
Trains the backbone on the 30k-subset × 35 concepts (Stage-1), then asks: do the concept-aware features **match-or-beat SSL features** on the NEW-CENTER metric (LOCO PPV@90R), apples-to-apples? This is the honest test that **updates the backbone** (not a frozen proxy, not a 35-scalar bottleneck).

Runtime → Change runtime type → **GPU**. Upload `rare26_concept.tar.gz` (from `python -m phase3.prepare_colab_pack`) to Drive. (For the FULL pipeline from a git clone, use `colab_full_pipeline.ipynb` instead.)

In [ ]:
!nvidia-smi -L
!pip -q install timm==1.0.27 scikit-learn

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!cp '/content/drive/MyDrive/rare26_concept.tar.gz' /content/ && tar -xzf /content/rare26_concept.tar.gz -C /content/
%cd /content/rare26_concept
import torch; print('cuda', torch.cuda.is_available()); !ls data && python -c "import numpy as np; d=np.load('data/concept_targets.npz',allow_pickle=True); print('concept frames', len(d['paths']))"

## Stage-1 — concept-supervised pretraining (curated core + GRL center-invariance)

In [ ]:
!python -m phase3.pretrain_concept --targets data/concept_targets.npz \
    --unfreeze 4 --epochs 12 --bs 64 --lr 1e-4 --grl 1.0 --out concept_encoder.pt

## Re-featurize the LABELED set with both backbones (SSL vs concept), then the FAIR gate

In [ ]:
# SSL baseline features and concept-encoder features over the SAME labeled frames
!python -m phase3.featurize --csv data/train.csv --out feats_train_ssl.npz     --workers 2
!python -m phase3.featurize --csv data/train.csv --out feats_train_concept.npz --workers 2 --backbone concept_encoder.pt

In [ ]:
# THE VERDICT: does concept-pretraining beat SSL on the new-center (LOCO) PPV@90R?
!python -m phase3.concept_gate --ssl feats_train_ssl.npz --concept feats_train_concept.npz --C 1.0

**If PASS** → the idea earns its place; run Stage-2 downstream fine-tune from the concept encoder and compare to FT-from-SSL:
```
!python -m phase3.finetune --holdout center_2 --init concept_encoder.pt --unfreeze 4 --epochs 30 --bs 64 --loss bce+rank+pauc --out ft_concept_c2.pt
!python -m phase3.finetune --holdout center_2                          --unfreeze 4 --epochs 30 --bs 64 --loss bce+rank+pauc --out ft_ssl_c2.pt
```
Compare the per-epoch `LOCO-val PPV@90R`. **If FAIL** → keep SSL; concepts → interpretability + positive-mining only. Either way, you have a real answer on YOUR idea.

Download the encoder: `from google.colab import files; files.download('concept_encoder.pt')`